In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parents[1]
OUTPUT_DIR = PROJECT_ROOT / "인벤데이터 시각화" / "outputs"
SOURCE_PATH = OUTPUT_DIR / "qna_anal.pkl"
RESULT_PATH = OUTPUT_DIR / "category_keyword_umap.csv"

# 프로젝트 RAG 파이프라인과 동일한 한국어 임베딩 모델
EMBEDDING_MODEL = "jhgan/ko-sroberta-multitask"

# 카테고리별로 표시할 키워드 수
TOP_N = 5

# 키워드 벡터를 만들 때 사용할 최대 게시글 수 (많을수록 느려진다)
MAX_POSTS_PER_KEYWORD = 40

# 임베딩에 사용할 게시글 본문 길이 상한
MAX_TEXT_LENGTH = 300

# 신규·복귀 집단을 나누는 기준 단어라 키워드 분석에서는 제외한다.
EXCLUDE_WORDS = {
    "뉴비",
    "메린이",
    "메린",
    "복귀",
    "복귀유저",
    "유입",
    "입문",
    "초보",
}

RANDOM_STATE = 42

In [ ]:
def load_posts() -> pd.DataFrame:
    """전처리된 질문 데이터를 불러온다."""
    if not SOURCE_PATH.is_file():
        raise FileNotFoundError(
            f"전처리 데이터를 찾지 못했습니다: {SOURCE_PATH}\n"
            "'인벤데이터 시각화/visualization.ipynb'를 먼저 실행해 주세요."
        )

    posts = pd.read_pickle(SOURCE_PATH)
    posts["post_text"] = (
        posts["title_clean"].fillna("").astype(str)
        + " "
        + posts["content_clean"].fillna("").astype(str)
    ).str.slice(0, MAX_TEXT_LENGTH)
    return posts


def clean_tokens(tokens) -> set:
    """게시글 하나의 토큰을 정리한다. 같은 단어는 한 번만 센다."""
    if not isinstance(tokens, (list, tuple, set, np.ndarray)):
        return set()

    return {
        str(token).strip()
        for token in tokens
        if isinstance(token, str)
        and len(token.strip()) >= 2
        and token.strip() not in EXCLUDE_WORDS
    }


def category_top_keywords(posts: pd.DataFrame) -> pd.DataFrame:
    """카테고리별로 등장 게시글 수가 많은 키워드 TOP N을 구한다."""
    rows = []

    for category, group in posts.groupby("category", sort=True):
        counter = Counter()
        for tokens in group["tokens"]:
            counter.update(clean_tokens(tokens))

        for rank, (word, count) in enumerate(counter.most_common(TOP_N), start=1):
            rows.append(
                {
                    "category": category,
                    "word": word,
                    "rank": rank,
                    "document_count": count,
                    "document_rate": count / len(group) * 100,
                    "category_post_count": len(group),
                }
            )

    return pd.DataFrame(rows)


def keyword_post_indexes(posts: pd.DataFrame, keywords: pd.DataFrame) -> dict:
    """(카테고리, 키워드)마다 해당 키워드가 등장한 게시글 index 목록을 모은다."""
    token_sets = posts["tokens"].map(clean_tokens)
    selection = {}

    for row in keywords.itertuples():
        in_category = posts["category"] == row.category
        has_word = token_sets.map(lambda words, w=row.word: w in words)
        matched = posts.index[in_category & has_word]

        # 게시글이 많은 키워드는 조회수가 높은 순으로 잘라 대표성을 유지한다.
        if len(matched) > MAX_POSTS_PER_KEYWORD:
            matched = (
                posts.loc[matched]
                .sort_values("views", ascending=False)
                .head(MAX_POSTS_PER_KEYWORD)
                .index
            )

        selection[(row.category, row.word)] = list(matched)

    return selection


def embed_keywords(posts: pd.DataFrame, selection: dict) -> tuple:
    """키워드별 게시글 임베딩 평균을 계산한다."""
    from sentence_transformers import SentenceTransformer

    needed = sorted({index for indexes in selection.values() for index in indexes})
    texts = posts.loc[needed, "post_text"].tolist()

    print(f"임베딩 대상 게시글 {len(texts):,}건 ({EMBEDDING_MODEL})")
    model = SentenceTransformer(EMBEDDING_MODEL)
    vectors = model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    position = {index: order for order, index in enumerate(needed)}

    keys = []
    matrix = []
    for key, indexes in selection.items():
        if not indexes:
            continue
        keys.append(key)
        matrix.append(vectors[[position[index] for index in indexes]].mean(axis=0))

    return keys, np.vstack(matrix)


def run_umap(matrix: np.ndarray) -> np.ndarray:
    """키워드 벡터를 UMAP 2차원 좌표로 줄인다."""
    from umap import UMAP

    reducer = UMAP(
        n_components=2,
        # 점이 30개뿐이라 이웃 수를 작게 잡아야 카테고리 단위 구조가 남는다.
        n_neighbors=min(8, len(matrix) - 1),
        # 같은 카테고리 점끼리 겹치지 않도록 넉넉히 벌려 놓는다.
        min_dist=0.8,
        spread=2.5,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    return reducer.fit_transform(matrix)


def main() -> None:
    posts = load_posts()
    keywords = category_top_keywords(posts)
    print(f"카테고리 {keywords['category'].nunique()}개 · 키워드 {len(keywords)}개")

    selection = keyword_post_indexes(posts, keywords)
    keys, matrix = embed_keywords(posts, selection)
    coordinates = run_umap(matrix)

    coordinate_frame = pd.DataFrame(
        {
            "category": [category for category, _ in keys],
            "word": [word for _, word in keys],
            "umap_x": coordinates[:, 0],
            "umap_y": coordinates[:, 1],
            "post_count": [len(selection[key]) for key in keys],
        }
    )

    result = keywords.merge(coordinate_frame, on=["category", "word"], how="inner")
    result = result.sort_values(["category", "rank"]).reset_index(drop=True)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    result.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {RESULT_PATH} ({len(result)}행)")